# Modelagem Gold (Esquema Estrela)

Duas dimensões conformadas (`dim_municipio`, `dim_tempo`) compartilhadas por três tabelas
fato, cada uma com seu grão e sua origem:

| Tabela | Grão | Origem |
|---|---|---|
| `fato_desempenho_educacional` | município × ano × etapa de ensino | IDEB (INEP) |
| `fato_infraestrutura_escolar` | município × ano | Microdados do Censo Escolar (INEP) |
| `fato_investimento_social` | município × ano | Novo Bolsa Família (Portal da Transparência) |

**A governança vem antes dos dados.** Cada tabela é declarada com `CREATE TABLE` explícito:
tipos, `NOT NULL`, descrição de cada coluna, chave primária, chaves estrangeiras, regras de
domínio (`CHECK`), propriedades e tags. Só depois os dados são inseridos, e a gravação falha
se violar alguma regra. Com isso o próprio Unity Catalog passa a ser o catálogo de dados do
modelo, incluindo o diagrama de relacionamento entre as tabelas.

Sobre as garantias: `NOT NULL` e `CHECK` são **aplicadas** pelo Delta em toda gravação. Chave
primária e estrangeira são **informativas** no Unity Catalog (documentam o modelo e ajudam o
otimizador, mas não bloqueiam gravação); a unicidade é verificada na Silver e a integridade
referencial, no fim deste notebook.

In [0]:
from pyspark.sql import functions as F

import os
import sys

# Funções compartilhadas entre os notebooks ficam em pipeline_utils.py, na mesma pasta.
_pasta = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
sys.path.insert(0, _pasta if _pasta.startswith("/Workspace") else f"/Workspace{_pasta}")
from pipeline_utils import aplicar_tags, texto_sql

# Parâmetro: usado pelo Job (Jobs & Pipelines) e com padrão para execução interativa.
dbutils.widgets.text("catalog", "workspace")

CATALOG = dbutils.widgets.get("catalog")

spark.sql(f"USE CATALOG {CATALOG}")


def criar_tabela_gold(tabela, descricao, colunas, chave_primaria, estrangeiras=(),
                      propriedades=None, checks=None, tags=None, tags_colunas=None):
    """Declara a tabela antes de qualquer dado: tipos, NOT NULL, descrição de cada coluna,
    chave primária, chaves estrangeiras, regras de domínio (CHECK) e tags.
    A carga vem depois e já tem de obedecer a esse contrato."""
    nome = f"{CATALOG}.gold.{tabela}"
    definicoes = [
        f"{coluna} {tipo}{'' if nula else ' NOT NULL'} COMMENT '{texto_sql(comentario)}'"
        for coluna, tipo, nula, comentario in colunas
    ]
    # RELY: a unicidade é garantida na Silver (garantir_chave_unica), então o otimizador
    # pode confiar na chave e eliminar agregações e junções redundantes.
    definicoes.append(f"CONSTRAINT pk_{tabela} PRIMARY KEY ({', '.join(chave_primaria)}) RELY")
    for nome_fk, colunas_fk, referencia, colunas_ref in estrangeiras:
        definicoes.append(
            f"CONSTRAINT {nome_fk} FOREIGN KEY ({', '.join(colunas_fk)}) "
            f"REFERENCES {CATALOG}.gold.{referencia} ({', '.join(colunas_ref)})"
        )
    props = ", ".join(f"'{k}' = '{texto_sql(v)}'" for k, v in (propriedades or {}).items())
    ddl = (f"CREATE OR REPLACE TABLE {nome} (\n  " + ",\n  ".join(definicoes) + "\n)\n"
           f"COMMENT '{texto_sql(descricao)}'\nTBLPROPERTIES ({props})")
    spark.sql(ddl)
    print(ddl, end="\n\n")

    # Regras de domínio: a partir daqui o Delta rejeita qualquer gravação que as viole.
    for nome_check, expressao in (checks or {}).items():
        spark.sql(f"ALTER TABLE {nome} ADD CONSTRAINT {nome_check} CHECK ({expressao})")
    aplicar_tags(nome, tags or {})
    for coluna, tags_coluna in (tags_colunas or {}).items():
        aplicar_tags(nome, tags_coluna, coluna)


def carregar(df, tabela):
    """Grava os dados respeitando a estrutura declarada, sem recriar a tabela."""
    nome = f"{CATALOG}.gold.{tabela}"
    df.createOrReplaceTempView("carga")
    colunas = ", ".join(spark.table(nome).columns)
    spark.sql(f"INSERT OVERWRITE {nome} SELECT {colunas} FROM carga")
    print(f"{tabela:32} {spark.table(nome).count():>8,} linhas carregadas")

## Reexecução segura

Numa segunda execução, as dimensões já existem e são referenciadas pelas chaves estrangeiras
dos fatos. Essas chaves são removidas antes de recriar as tabelas e voltam a ser declaradas
junto com os fatos logo abaixo.

In [0]:
CHAVES_ESTRANGEIRAS = {
    "fato_desempenho_educacional": ["fk_desempenho_municipio", "fk_desempenho_tempo"],
    "fato_infraestrutura_escolar": ["fk_infraestrutura_municipio", "fk_infraestrutura_tempo"],
    "fato_investimento_social": ["fk_investimento_municipio", "fk_investimento_tempo"],
}

for fato, restricoes in CHAVES_ESTRANGEIRAS.items():
    if spark.catalog.tableExists(f"{CATALOG}.gold.{fato}"):
        for restricao in restricoes:
            spark.sql(f"ALTER TABLE {CATALOG}.gold.{fato} DROP CONSTRAINT IF EXISTS {restricao}")

## Dimensões

In [0]:
UFS = "'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS','MG','PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO'"
REGIOES = "'Norte','Nordeste','Centro-Oeste','Sudeste','Sul'"

criar_tabela_gold(
    "dim_municipio",
    "Dimensão conformada de municípios brasileiros, compartilhada pelos três fatos. Fonte: IBGE.",
    colunas=[
        ("codigo_municipio_ibge", "BIGINT", False, "Código IBGE do município, 7 dígitos. Chave da dimensão."),
        ("nome_municipio", "STRING", False, "Nome oficial do município (IBGE)."),
        ("sigla_uf", "STRING", False, "Sigla da unidade da federação. Domínio: as 27 UFs."),
        ("nome_regiao", "STRING", False, "Região geográfica. Domínio: Norte, Nordeste, Centro-Oeste, Sudeste, Sul."),
        ("populacao", "BIGINT", True, "População residente no Censo 2022. Nula para municípios instalados depois do Censo, como Boa Esperança do Norte (MT)."),
    ],
    chave_primaria=["codigo_municipio_ibge"],
    propriedades={"camada": "gold", "tipo_tabela": "dimensao", "grao": "municipio", "origem": "silver.municipios"},
    checks={
        "uf_valida": f"sigla_uf IN ({UFS})",
        "regiao_valida": f"nome_regiao IN ({REGIOES})",
        "populacao_positiva": "populacao IS NULL OR populacao > 0",
    },
    tags={"camada": "gold", "tipo_tabela": "dimensao", "dominio": "territorio", "fonte": "ibge", "dados_pessoais": "nao"},
    tags_colunas={
        "codigo_municipio_ibge": {"papel": "chave_primaria"},
        "populacao": {"papel": "atributo", "unidade": "pessoas"},
    },
)

criar_tabela_gold(
    "dim_tempo",
    "Dimensão de tempo com os anos cobertos pelas fontes: IDEB 2023, Censo Escolar 2023 e Bolsa Família 2024.",
    colunas=[
        ("ano", "INT", False, "Ano de referência. Chave da dimensão."),
        ("decada", "INT", False, "Década do ano de referência."),
    ],
    chave_primaria=["ano"],
    propriedades={"camada": "gold", "tipo_tabela": "dimensao", "grao": "ano"},
    checks={"ano_plausivel": "ano BETWEEN 2000 AND 2100"},
    tags={"camada": "gold", "tipo_tabela": "dimensao", "dominio": "tempo", "fonte": "derivada", "dados_pessoais": "nao"},
    tags_colunas={"ano": {"papel": "chave_primaria"}},
)

## Fatos

In [0]:
def chaves_do_fato(prefixo):
    return [
        (f"fk_{prefixo}_municipio", ["codigo_municipio_ibge"], "dim_municipio", ["codigo_municipio_ibge"]),
        (f"fk_{prefixo}_tempo", ["ano"], "dim_tempo", ["ano"]),
    ]


TAGS_CHAVES = {
    "codigo_municipio_ibge": {"papel": "chave_estrangeira"},
    "ano": {"papel": "chave_estrangeira"},
}

criar_tabela_gold(
    "fato_desempenho_educacional",
    "IDEB 2023 e proficiências do SAEB da rede pública. Grão: município x ano x etapa de ensino. Fonte: INEP.",
    colunas=[
        ("codigo_municipio_ibge", "BIGINT", False, "Município. Referencia dim_municipio."),
        ("ano", "INT", False, "Ano da edição do IDEB. Referencia dim_tempo."),
        ("etapa_ensino", "STRING", False, "Etapa do ensino fundamental. Domínio: Anos Iniciais, Anos Finais."),
        ("vl_ideb", "DOUBLE", True, "IDEB observado da rede pública. Escala de 0 a 10. Nulo quando o município não teve IDEB calculado na etapa."),
        ("vl_nota_matematica", "DOUBLE", True, "Proficiência média em Matemática no SAEB. Escala SAEB (entre 134 e 405 nos dados de 2023)."),
        ("vl_nota_portugues", "DOUBLE", True, "Proficiência média em Língua Portuguesa no SAEB. Escala SAEB."),
        ("vl_indicador_rendimento", "DOUBLE", True, "Indicador de rendimento, calculado a partir das taxas de aprovação. Escala de 0 a 1."),
    ],
    chave_primaria=["codigo_municipio_ibge", "ano", "etapa_ensino"],
    estrangeiras=chaves_do_fato("desempenho"),
    propriedades={"camada": "gold", "tipo_tabela": "fato", "grao": "municipio_ano_etapa", "origem": "silver.ideb"},
    checks={
        "etapa_valida": "etapa_ensino IN ('Anos Iniciais', 'Anos Finais')",
        "ideb_na_escala": "vl_ideb IS NULL OR vl_ideb BETWEEN 0 AND 10",
        "saeb_na_escala": "(vl_nota_matematica IS NULL OR vl_nota_matematica BETWEEN 0 AND 500) "
                          "AND (vl_nota_portugues IS NULL OR vl_nota_portugues BETWEEN 0 AND 500)",
        "rendimento_na_escala": "vl_indicador_rendimento IS NULL OR vl_indicador_rendimento BETWEEN 0 AND 1",
    },
    tags={"camada": "gold", "tipo_tabela": "fato", "dominio": "educacao", "fonte": "inep", "dados_pessoais": "nao"},
    tags_colunas={
        **TAGS_CHAVES,
        "vl_ideb": {"papel": "metrica", "unidade": "indice_0_10"},
        "vl_nota_matematica": {"papel": "metrica", "unidade": "escala_saeb"},
        "vl_nota_portugues": {"papel": "metrica", "unidade": "escala_saeb"},
        "vl_indicador_rendimento": {"papel": "metrica", "unidade": "indice_0_1"},
    },
)

In [0]:
# As colunas de percentual e suas descrições vêm da Silver, que as herdou do dicionário
# oficial do INEP: nenhuma descrição é redigida duas vezes.
colunas_pct = [
    (f.name, "DOUBLE", False, f.metadata.get("comment", f.name))
    for f in spark.table(f"{CATALOG}.silver.censo_escolar_infra").schema.fields
    if f.name.startswith("pct_")
]

criar_tabela_gold(
    "fato_infraestrutura_escolar",
    "Infraestrutura das escolas públicas em atividade, agregada por município. Grão: município x ano. Fonte: Censo Escolar 2023 (INEP).",
    colunas=[
        ("codigo_municipio_ibge", "BIGINT", False, "Município. Referencia dim_municipio."),
        ("ano", "INT", False, "Ano do Censo Escolar. Referencia dim_tempo."),
        ("qt_escolas_publicas", "BIGINT", False, "Escolas públicas (federal, estadual e municipal) em atividade no município."),
        ("qt_matriculas", "BIGINT", True, "Matrículas da educação básica nessas escolas."),
        ("qt_docentes", "BIGINT", True, "Docentes da educação básica nessas escolas. Quem leciona em mais de uma escola é contado em cada uma."),
        *colunas_pct,
        ("media_alunos_por_escola", "DOUBLE", True, "Matrículas divididas pelo número de escolas públicas em atividade."),
        ("media_alunos_por_docente", "DOUBLE", True, "Matrículas divididas pelo número de docentes."),
    ],
    chave_primaria=["codigo_municipio_ibge", "ano"],
    estrangeiras=chaves_do_fato("infraestrutura"),
    propriedades={"camada": "gold", "tipo_tabela": "fato", "grao": "municipio_ano", "origem": "silver.censo_escolar_infra"},
    checks={
        "escolas_positivas": "qt_escolas_publicas > 0",
        "medias_nao_negativas": "(media_alunos_por_escola IS NULL OR media_alunos_por_escola >= 0) "
                                "AND (media_alunos_por_docente IS NULL OR media_alunos_por_docente >= 0)",
        **{f"{nome}_na_escala": f"{nome} BETWEEN 0 AND 100" for nome, *_ in colunas_pct},
    },
    tags={"camada": "gold", "tipo_tabela": "fato", "dominio": "educacao", "fonte": "inep", "dados_pessoais": "nao"},
    tags_colunas={
        **TAGS_CHAVES,
        "qt_escolas_publicas": {"papel": "metrica", "unidade": "escolas"},
        "qt_matriculas": {"papel": "metrica", "unidade": "matriculas"},
        "qt_docentes": {"papel": "metrica", "unidade": "docentes"},
        **{nome: {"papel": "metrica", "unidade": "percentual"} for nome, *_ in colunas_pct},
    },
)

In [0]:
criar_tabela_gold(
    "fato_investimento_social",
    "Novo Bolsa Família no mês de referência, absoluto e normalizado pela população. Grão: município x ano. Fonte: Portal da Transparência.",
    colunas=[
        ("codigo_municipio_ibge", "BIGINT", False, "Município. Referencia dim_municipio."),
        ("ano", "INT", False, "Ano do mês de referência. Referencia dim_tempo."),
        ("mes_referencia", "STRING", False, "Mês de competência dos pagamentos (AAAAMM)."),
        ("valor_total", "DOUBLE", False, "Soma das parcelas pagas no município, em R$. Não usar para comparar municípios: reflete o tamanho da população."),
        ("quantidade_beneficiados", "BIGINT", False, "Quantidade de benefícios pagos no município."),
        ("valor_per_capita", "DOUBLE", True, "valor_total dividido pela população do Censo 2022, em R$ por habitante. Métrica indicada para comparar municípios."),
        ("taxa_cobertura_pct", "DOUBLE", True, "Benefícios pagos por 100 habitantes. Aproxima o alcance do programa no município."),
    ],
    chave_primaria=["codigo_municipio_ibge", "ano"],
    estrangeiras=chaves_do_fato("investimento"),
    propriedades={"camada": "gold", "tipo_tabela": "fato", "grao": "municipio_ano",
                  "origem": "silver.bolsa_familia + gold.dim_municipio", "mes_referencia": "202405"},
    checks={
        "valores_nao_negativos": "valor_total >= 0 AND quantidade_beneficiados >= 0",
        "per_capita_nao_negativo": "valor_per_capita IS NULL OR valor_per_capita >= 0",
        "cobertura_na_escala": "taxa_cobertura_pct IS NULL OR taxa_cobertura_pct BETWEEN 0 AND 100",
    },
    tags={"camada": "gold", "tipo_tabela": "fato", "dominio": "assistencia_social",
          "fonte": "portal_transparencia", "dados_pessoais": "nao", "anonimizado": "sim"},
    tags_colunas={
        **TAGS_CHAVES,
        "valor_total": {"papel": "metrica", "unidade": "reais"},
        "quantidade_beneficiados": {"papel": "metrica", "unidade": "beneficios"},
        "valor_per_capita": {"papel": "metrica", "unidade": "reais_por_habitante"},
        "taxa_cobertura_pct": {"papel": "metrica", "unidade": "percentual"},
    },
)

## Carga

Com as cinco tabelas declaradas, os dados entram por `INSERT OVERWRITE`, que preserva a
estrutura, as descrições e as regras. Uma linha que viole `NOT NULL` ou `CHECK` faz a carga
falhar em vez de gravar dado inválido.

In [0]:
carregar(spark.table(f"{CATALOG}.silver.municipios"), "dim_municipio")

anos = (
    spark.table(f"{CATALOG}.silver.ideb").select("ano")
    .union(spark.table(f"{CATALOG}.silver.censo_escolar_infra").select("ano"))
    .union(spark.table(f"{CATALOG}.silver.bolsa_familia").select("ano"))
    .distinct()
)
carregar(anos.withColumn("decada", (F.col("ano") / 10).cast("int") * 10), "dim_tempo")

carregar(spark.table(f"{CATALOG}.silver.ideb"), "fato_desempenho_educacional")

carregar(
    spark.table(f"{CATALOG}.silver.censo_escolar_infra")
    .withColumn("qt_matriculas", F.col("qt_matriculas").cast("bigint"))
    .withColumn("qt_docentes", F.col("qt_docentes").cast("bigint"))
    .withColumn("media_alunos_por_escola", F.round(F.col("qt_matriculas") / F.col("qt_escolas_publicas"), 1))
    .withColumn("media_alunos_por_docente", F.round(F.col("qt_matriculas") / F.col("qt_docentes"), 1)),
    "fato_infraestrutura_escolar",
)

carregar(
    spark.table(f"{CATALOG}.silver.bolsa_familia")
    .join(spark.table(f"{CATALOG}.gold.dim_municipio").select("codigo_municipio_ibge", "populacao"),
          "codigo_municipio_ibge", "left")
    .withColumn("valor_per_capita", F.round(F.col("valor_total") / F.col("populacao"), 2))
    .withColumn("taxa_cobertura_pct", F.round(F.col("quantidade_beneficiados") / F.col("populacao") * 100, 2)),
    "fato_investimento_social",
)

## Validação de integridade referencial

Como as chaves estrangeiras do Unity Catalog são informativas, a integridade é conferida aqui:
todo município e todo ano presentes num fato têm de existir na dimensão correspondente.

In [0]:
for fato in CHAVES_ESTRANGEIRAS:
    df = spark.table(f"{CATALOG}.gold.{fato}")
    for coluna, dimensao in [("codigo_municipio_ibge", "dim_municipio"), ("ano", "dim_tempo")]:
        orfaos = df.select(coluna).distinct().join(
            spark.table(f"{CATALOG}.gold.{dimensao}").select(coluna), coluna, "left_anti"
        ).count()
        assert orfaos == 0, f"{fato}.{coluna}: {orfaos} valores sem correspondência em {dimensao}"
    print(f"{fato:32} {df.count():>8,} linhas | integridade OK")

## O catálogo publicado

Tudo o que foi declarado acima fica consultável no `information_schema`, e é o mesmo conteúdo
que o Catalog Explorer exibe. Estas consultas são a base do Catálogo de Dados do README.

In [0]:
display(spark.sql(f"""
    SELECT table_name, column_name, data_type, is_nullable, comment
    FROM {CATALOG}.information_schema.columns
    WHERE table_schema = 'gold'
    ORDER BY table_name, ordinal_position
"""))

In [0]:
display(spark.sql(f"""
    SELECT table_name, constraint_name, constraint_type
    FROM {CATALOG}.information_schema.table_constraints
    WHERE table_schema = 'gold'
    ORDER BY table_name, constraint_type
"""))

display(spark.sql(f"""
    SELECT table_name, tag_name, tag_value
    FROM {CATALOG}.information_schema.table_tags
    WHERE schema_name = 'gold'
    ORDER BY table_name, tag_name
"""))